In [ ]:
# # Transport Atlas 데이터셋의 장점과 단점
# 제공해 주신 프로파일링, 패치, 타당성 점검, 회귀, OpenAlex 재검증 결과를 종합해 평가했습니다.

# ## 장점
# **1. 분야 단위의 포괄성과 시간 깊이**
# - 교통 분야 34개 저널, 1967~2026년(약 60년), 논문 122,669행(분석 표본 118,985편)으로 한 분야를 거의 전수 관측합니다. 단일 이슈가 아니라 분야 전체의 구조 변화를 볼 수 있습니다.
# - 2020년대 논문이 많아(2020~2025년 약 6만 편) 최근 현상(ML 확산 등)을 분석하기에 표본이 충분합니다.
# **2. 서로 연결 가능한 다층 구조**
# - 논문, 논문×저자, 저자, 저자×연도, 공저 엣지, 저널×연도, 커뮤니티, 임베딩의 9개 파일이 참조 무결성 검사를 통과했습니다.
# - SPECTER2 임베딩(95,166편)으로 주제 거리, 참신성, 주제 이동, 토픽 군집을 만들 수 있고, 네트워크 구조(공저, 커뮤니티)와 결합할 수 있습니다. 키워드만으로는 어려운 분석이 가능합니다.
# **2-1. 보정 후 매칭 품질이 높음**
# - 이름 형식을 정규화하자 저자 매칭이 0.8%에서 unique 60.8%로, 네트워크 저자 연결이 91.9%로 올랐습니다. `n_papers` 대 `n_papers_obs` Spearman은 0.959입니다.
# **3. 외부 데이터와의 결합 가능성**
# - DOI 보유율이 약 90%이고, OpenAlex 회수율이 99.8%였습니다. `counts_by_year`로 출판 후 2~3년 고정 창 인용을 만들 수 있었고, 저자 ID, 토픽 분류도 붙일 수 있었습니다.
# - 저자 수 일치율이 99.9%라 OpenAlex 결합의 신뢰도가 높습니다.
# **4. 방법론적 점검이 가능한 규모**
# - 8개 변형 강건성 점검, 순열 검정, 대조군(`bayesian`), 시기 분할 같은 검증을 돌릴 만큼 표본이 큽니다. 여러 가설을 기각하고 가려낼 수 있었다는 점이 데이터의 힘입니다.

# ## 단점
# **1. 저자 식별의 약점 (가장 큼)**
# - 이름 기반 매칭이라 동명이인과 표기 변이 오류가 남습니다. `ambiguous` 7%, 위험군 제외 처리가 필요했고, 완전 매칭은 논문의 35.5%에 그칩니다.
# - 해결 방향: OpenAlex 저자 ID로 교체.
# **2. 코퍼스 내부로 한정된 관측**
# - 저자의 다른 저널 이력이 보이지 않아, "ML 경험자", "저자 질" 같은 변수가 측정오차를 크게 갖고, `new_entrant`나 `converter`에도 실제 경험자가 섞입니다.
# - 영어권 34개 저널 중심이라 지역 언어 저널과 학회 논문이 빠집니다.
# **3. 인용 변수의 한계**
# - `cited_by_count`는 수집 시점 누적이라 논문 나이에 민감합니다. 이는 고정 창 인용(OpenAlex)으로 보완해야 했고, 2023년 이후 논문은 창이 짧거나 비어 있습니다(`cit3_z` 유효 약 50%).
# - 인용 기반 저자 질 통제도 시점상 엄밀한 사전 변수가 아닙니다.
# **4. 임베딩 기반 변수의 구조적 문제**
# - 보유율이 78.8%이고 초기 연도일수록 낮습니다. 저자당 30편 상한도 있습니다.
# - SPECTER2는 인용으로 학습된 모델이라 `ml_score`, `novelty`, `cit_z` 사이에 순환적 상관이 생길 수 있습니다. 실제로 `ml_z` 계수는 해석하지 말라고 판단했습니다.
# - ML 축은 대조군 `bayesian` AUC 0.718로 순수 ML만 잡지 못했습니다.
# **5. 일부 변수의 품질 결함**
# - `comm_combined`는 의미 커뮤니티와 거의 무관하고(NMI 0.018) 원인이 미해결이라 핵심 변수로 쓸 수 없습니다. `reviewer_index`의 개념 프로필은 코퍼스 전 기간 집계라 도입 이후 활동이 섞여 사용하지 못했습니다.
# - 비연구 항목(2,165건)과 제목 중복(1,519건)은 플래그로 처리했지만, 분석마다 `in_sample` 확인이 필요합니다.
# **6. 인과 식별 요소 부재**
# - 정책, 기관, 펀딩 같은 외생 변동이 없어 거의 모든 분석이 관찰 연구입니다. 실제로 전문가 합류 프리미엄은 저자 질 통제 후 절반으로 줄었습니다(선택효과).
# - 단일 분야라 일반화에 한계가 있습니다.
# **7. 이벤트 표본이 작아지는 문제**
# - 전체 규모는 크지만, 세부 조건(초기 채택자, 팀 유형, 경력×거리 셀)을 걸면 셀이 10~30건대로 줄어 단계 분석이 불가능해졌습니다. 최종 분석 표본이 1,328편 수준이었습니다.

# ## 종합
# | 구분 | 평가 |
# |---|---|
# | 강점 | 분야 전수 관측, 60년 시계열, 임베딩+네트워크+외부 결합 |
# | 약점 | 저자 식별, 코퍼스 외부 이력 부재, 인용·임베딩의 측정 문제, 외생 변동 없음 |
# | 적합한 용도 | 분야 구조 변화, 주제 구성, 저널·시기별 추세 등 **집계 수준의 기술적·구조적 분석** |
# | 부적합한 용도 | 저자 단위 인과 추정, 미세한 단계 분석 |
# 실제 결과도 이를 뒷받침합니다. 저자 단위 인과 가설(네트워크 거리, 전문가 합류, 주제 이동)은 대부분 기각되거나 약해졌고, 논문·저널·주제 단위의 구성 변화 분석(예: 안전 비중 하락의 between 효과 분해)은 비교적 명확한 결과를 냈습니다. 이 데이터는 **저자 개인을 추적하는 연구보다 분야 자체를 측정하는 연구**에서 더 강합니다.

In [6]:
# ============================================================
# 사전 탐색 v2 (단일 셀, 외부 수집 없음)
# [1] 코퍼스 구성  [2] 주제 구조·저널 분화(전체 vs 고정 저널)  [3] 신생·쇠퇴 주제
# [4] 갱신 메커니즘: 전환 vs 교체  [5] 공저 네트워크 연결성  [6] 참신성-인용  [7] 판정표
# 입력 : analysis_data/{papers_master, paper_ml_features, paper_author_long}.csv
# 출력 : analysis_data/profile4/*.csv
# ============================================================
import re, warnings, traceback
from collections import Counter
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT = Path("./analysis_data"); P4 = OUT / "profile4"; P4.mkdir(exist_ok=True)
hdr = lambda t: print("\n" + "=" * 95 + f"\n{t}\n" + "=" * 95)
RES = {}
def safe(name, fn):
    try: fn()
    except Exception: print(f"✗ [{name}] 실패"); traceback.print_exc()

# ---------------- 데이터 ----------------
pm = pd.read_csv(OUT / "papers_master.csv", usecols=["pid", "title", "year", "venue_short", "in_sample", "n_authors", "is_solo"])
ft = pd.read_csv(OUT / "paper_ml_features.csv", usecols=["pid", "has_emb", "topic150", "novelty_prior5", "cit_z", "ml98"])
D = pm[pm.in_sample.astype(bool) & (pm.year <= 2025)].merge(ft, on="pid", how="left")   # 2026 부분연도 제외
D["has_emb"] = D.has_emb.fillna(False).astype(bool); D["ml98"] = D.ml98.fillna(False).astype(bool)
D["period"] = np.minimum(D.year // 5 * 5, 2020)           # ..., 2015-19, 2020-25
PERS = [2000, 2005, 2010, 2015, 2020]
K = int(D.topic150.max()) + 1
npp = D[D.year >= 2000].groupby(["venue_short", "period"]).size().unstack(fill_value=0)
FIX = set(npp.index[(npp[PERS] >= 50).all(axis=1)])         # 모든 기간에 50편 이상 있는 저널
print(f"논문 {len(D):,} | topic150 보유 {D.topic150.notna().mean():.1%} | 고정 저널 {len(FIX)}개 / 전체 {D.venue_short.nunique()}개")

def tdist(df):
    c = np.bincount(df.topic150.dropna().astype(int), minlength=K).astype(float); return c / c.sum() if c.sum() else c
def H(p): p = p[p > 0]; return -(p * np.log(p)).sum()
def js(p, q):
    m = .5 * (p + q); kl = lambda a, b: np.sum(np.where(a > 0, a * np.log(a / np.where(b > 0, b, 1)), 0))
    return .5 * kl(p, m) + .5 * kl(q, m)
def nmi_corr(df):                                            # 저널-주제 상호정보(표본 편향 보정, Miller-Madow)
    t = pd.crosstab(df.venue_short, df.topic150.astype(int)); N = t.values.sum()
    p = t.values / N; pv, pt = p.sum(1, keepdims=True), p.sum(0, keepdims=True)
    nz = p > 0; mi = (p[nz] * np.log(p[nz] / (pv @ pt)[nz])).sum()
    mi -= (t.shape[0] - 1) * (t.shape[1] - 1) / (2 * N)
    return mi / H(pt.ravel())

# ============================================================
# [1] 코퍼스 구성
# ============================================================
def sec1():
    hdr("[1] 코퍼스 구성 (기간별)")
    g = D[D.year >= 1990].assign(p=lambda d: np.minimum(d.year // 5 * 5, 2020)).groupby("p")
    t = pd.DataFrame({"papers": g.size(), "venues(>=20편)": g.venue_short.apply(lambda s: (s.value_counts() >= 20).sum()),
                      "임베딩 보유": g.has_emb.mean(), "고정저널 비중": g.venue_short.apply(lambda s: s.isin(FIX).mean()),
                      "평균 팀규모": g.n_authors.mean(), "단독저자": g.is_solo.mean()})
    print(t.to_string()); t.to_csv(P4 / "corpus_by_period.csv")
    print("※ 임베딩 보유율이 기간별로 달라서 이후 주제 분석은 2000년 이후만 씁니다.")
safe("sec1", sec1)

# ============================================================
# [2] 주제 구조와 저널 분화 (전체 vs 고정 저널)
# ============================================================
def sec2():
    hdr("[2] 주제 다양성·회전·저널 분화 (전체 / 고정 저널)")
    rows = []
    for lab, sub in [("전체", D), ("고정저널", D[D.venue_short.isin(FIX)])]:
        prev = None
        for p in PERS:
            d = sub[(sub.period == p) & sub.topic150.notna()]; s = tdist(d)
            rows.append(dict(set=lab, period=p, N=len(d), eff_topics=np.exp(H(s)), HHI=(s ** 2).sum(),
                             JS_vs_prev=js(prev, s) if prev is not None else np.nan, venue_topic_NMI=nmi_corr(d)))
            prev = s
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(P4 / "topic_structure.csv", index=False)
    for lab in ["전체", "고정저널"]:
        a = T[(T.set == lab)].set_index("period")
        RES[f"effN_chg_{lab}"] = a.eff_topics[2020] / a.eff_topics[2000] - 1
        RES[f"nmi_chg_{lab}"] = a.venue_topic_NMI[2020] - a.venue_topic_NMI[2000]
    print("\n※ NMI는 표본 편향을 보정했지만 기간별 N이 달라 절대값보다 방향·크기 비교용입니다.")
    rk = [spearmanr(tdist(D[(D.period == a) & D.topic150.notna()]), tdist(D[(D.period == b) & D.topic150.notna()]))[0]
          for a, b in zip(PERS[:-1], PERS[1:])]
    print("연속 기간 주제 비중 순위상관(높을수록 구조가 안정):", np.round(rk, 3))
safe("sec2", sec2)

# ============================================================
# [3] 신생·쇠퇴 주제
# ============================================================
EM = set(); LAB = {}
STOP = set("with from using based study analysis model models approach method methods case data their this that into under over towards toward between during traffic transport transportation".split())
def sec3():
    global EM, LAB
    hdr("[3] 신생·쇠퇴 주제 (2005-09 vs 2020-25)")
    d0 = D[(D.year.between(2005, 2009)) & D.topic150.notna()]; d1 = D[(D.year >= 2020) & D.topic150.notna()]
    s0, s1 = tdist(d0), tdist(d1)
    R = pd.DataFrame({"s0": s0, "s1": s1}); R["ratio"] = (R.s1 + 1e-4) / (R.s0 + 1e-4); R["delta"] = R.s1 - R.s0
    EM = set(R[(R.ratio >= 3) & (R.s1 >= .005)].index); DECL = set(R[(R.ratio <= 1 / 3) & (R.s0 >= .005)].index)
    RES["n_emerging"], RES["n_declining"] = len(EM), len(DECL)
    print(f"신생(비중 3배↑, 현재 0.5%↑) {len(EM)}개 | 쇠퇴(1/3↓, 과거 0.5%↑) {len(DECL)}개 | 전체 {K}개")
    E = D[D.topic150.notna() & (D.year >= 2005)].copy(); tok = lambda s: set(w for w in re.findall(r"[a-z]{4,}", str(s).lower()) if w not in STOP)
    E["tk"] = E.title.map(tok); n_all = len(E); ca = Counter(w for s in E.tk for w in s)
    def words(k):
        g = E[E.topic150 == k]; c = Counter(w for s in g.tk for w in s)
        sc = {w: (v / len(g)) / (ca[w] / n_all) for w, v in c.items() if v >= 10}
        return ", ".join(sorted(sc, key=sc.get, reverse=True)[:6])
    for k in R.index: LAB[k] = None
    for nm, S in [("신생", EM), ("쇠퇴", DECL)]:
        t = R.loc[list(S)].sort_values("delta", ascending=(nm == "쇠퇴")).head(8)
        print(f"\n[{nm}] 상위 8개 (topic: s0 → s1 | 대표 단어)")
        for k, r in t.iterrows(): LAB[k] = words(k); print(f"  {k:>3}: {r.s0:.3%} → {r.s1:.3%} | {LAB[k]}")
    R.to_csv(P4 / "topic_change.csv")
safe("sec3", sec3)

# ============================================================
# [4] 갱신 메커니즘: 전환(conversion) vs 교체(replacement)
# ============================================================
def sec4():
    hdr("[4] 주제 변화의 원천: 기존 저자의 전환 vs 신규·퇴출 교체 (이름 기반, 동명이인 위험군 제외)")
    pal = pd.read_csv(OUT / "paper_author_long.csv", usecols=["pid", "name_norm", "match_status", "year", "in_sample"], low_memory=False)
    pal = pal[pal.in_sample.astype(bool) & pal.name_norm.notna() & pal.year.notna()].drop_duplicates(["pid", "name_norm"]).copy()
    pal["year"] = pal.year.astype(int); pal["w"] = 1.0 / pal.groupby("pid").name_norm.transform("size")
    amb = set(pal.loc[pal.match_status == "ambiguous", "name_norm"])
    heavy = set(pal.groupby(["name_norm", "year"]).size().groupby("name_norm").max().loc[lambda s: s >= 12].index)
    pal = pal[~pal.name_norm.isin(amb | heavy)]
    first = pal.groupby("name_norm").year.min()
    pt = pal.merge(D[["pid", "topic150"]], on="pid").dropna(subset=["topic150"]); pt["topic150"] = pt.topic150.astype(int)
    for (a0, a1), (b0, b1), tag in [((2010, 2014), (2020, 2024), "주 분석 P0=2010-14, P1=2020-24"),
                                    ((2005, 2009), (2015, 2019), "견고성 P0=2005-09, P1=2015-19")]:
        w0, w1 = pt[pt.year.between(a0, a1)], pt[pt.year.between(b0, b1)]
        A0, A1 = set(w0.name_norm), set(w1.name_norm); cont, ent, ext = A0 & A1, A1 - A0, A0 - A1
        mv = lambda df, S: (lambda d: np.bincount(d.topic150, weights=d.w, minlength=K))(df[df.name_norm.isin(S)])
        m1c, m1e, m0c, m0x = mv(w1, cont), mv(w1, ent), mv(w0, cont), mv(w0, ext)
        M1, M0 = m1c + m1e, m0c + m0x; S1, S0 = M1 / M1.sum(), M0 / M0.sum()
        S1c, S0c, S1e, S0x = m1c / m1c.sum(), m0c / m0c.sum(), m1e / m1e.sum(), m0x / m0x.sum()
        w1c = m1c.sum() / M1.sum()
        delta = S1 - S0; conv = w1c * (S1c - S0c); repl = delta - conv           # 둘 다 합이 0
        tv = lambda v: .5 * np.abs(v).sum(); sh = lambda v: float(v @ delta / (delta @ delta))
        print(f"\n--- {tag} ---")
        print(f"저자: 지속 {len(cont):,} | 신규(P1만) {len(ent):,} | 퇴출(P0만) {len(ext):,}")
        print(f"P1 논문 질량 중 지속 저자 {w1c:.1%} | 신규 {1 - w1c:.1%} (그중 첫 논문이 {b0 - 5}년 이후인 진짜 신규 "
              f"{pt[pt.year.between(b0, b1) & pt.name_norm.isin(ent) & (pt.name_norm.map(first) >= b0 - 5)].w.sum() / M1.sum():.1%})")
        print(f"주제 분포 이동 JS(S0,S1) {js(S0, S1):.4f} | 지속저자 내부 JS(S0c,S1c) {js(S0c, S1c):.4f} | "
              f"신규 대 퇴출 JS(S1e,S0x) {js(S1e, S0x):.4f} | 선택효과 JS(S0c,S0x) {js(S0c, S0x):.4f}")
        print(f"변화량 TV: 전체 {tv(delta):.4f} = 전환 성분 {tv(conv):.4f} + 교체 성분 {tv(repl):.4f}")
        print(f"변화에 대한 기여 비중(투영): 전환 {sh(conv):.1%} | 교체 {sh(repl):.1%}  (합 100%)")
        if EM:
            em = list(EM); f = lambda df, S: df[df.name_norm.isin(S)].assign(e=lambda d: d.topic150.isin(em)).groupby("name_norm").e.max()
            e0 = f(w0, cont); e1c = f(w1, cont); e1n = f(w1, ent)
            print(f"신생 주제 참여율: P1 지속저자 {e1c.mean():.1%} (그중 P0에 신생 주제 없던 저자의 신규 진입 {e1c[~e0.reindex(e1c.index).fillna(False)].mean():.1%}) | P1 신규 저자 {e1n.mean():.1%}")
        if tag.startswith("주"):
            RES["conv_share"], RES["repl_share"], RES["n_cont"], RES["w1c"] = sh(conv), sh(repl), len(cont), w1c
            top = np.argsort(-np.abs(delta))[:8]
            print("\n변화 큰 주제 8개: Δ, 전환 기여, 교체 기여 | 대표 단어")
            for k in top: print(f"  {k:>3}: {delta[k]:+.4f}, {conv[k]:+.4f}, {repl[k]:+.4f} | {LAB.get(k) or ''}")
    print("\n※ 지속 저자는 두 기간 모두 논문이 있는 다작 저자로 선택되어 있고, 이름 기반이라 동명이인이 지속 저자를 부풀릴 수 있습니다.")
safe("sec4", sec4)

# ============================================================
# [5] 공저 네트워크 연결성 (기간별 거대 성분)
# ============================================================
def sec5():
    hdr("[5] 공저 네트워크: 2인 이상 논문의 저자 중 최대 연결 성분 비중")
    pal = pd.read_csv(OUT / "paper_author_long.csv", usecols=["pid", "name_norm", "match_status", "year", "in_sample"], low_memory=False)
    pal = pal[pal.in_sample.astype(bool) & pal.name_norm.notna() & pal.year.notna()].drop_duplicates(["pid", "name_norm"])
    pal = pal[~pal.name_norm.isin(set(pal.loc[pal.match_status == "ambiguous", "name_norm"]))]
    rows = []
    for y0, y1 in [(1990, 1994), (1995, 1999), (2000, 2004), (2005, 2009), (2010, 2014), (2015, 2019), (2020, 2025)]:
        d = pal[pal.year.between(y0, y1)]; d = d[d.groupby("pid").name_norm.transform("size") >= 2]
        if len(d) == 0: continue
        ac, pc = pd.factorize(d.name_norm)[0], pd.factorize(d.pid)[0]; na, npap = ac.max() + 1, pc.max() + 1
        G = sp.coo_matrix((np.ones(len(d)), (ac, na + pc)), shape=(na + npap, na + npap))
        _, lab = connected_components(G, directed=False); sz = np.bincount(lab[:na])
        rows.append(dict(window=f"{y0}-{y1}", authors=na, papers=npap, LCC_share=sz.max() / na, n_components=len(sz)))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(P4 / "lcc.csv", index=False)
    RES["lcc_first"], RES["lcc_last"] = T.LCC_share.iloc[0], T.LCC_share.iloc[-1]
    print("※ 코퍼스가 34개 저널로 한정되어 거대 성분은 '분야 전체'가 아니라 '이 저널 집합 안'의 연결성입니다.")
safe("sec5", sec5)

# ============================================================
# [6] 참신성-인용 관계 (임베딩 보유 논문, 저널·연도 내 순위)
# ============================================================
def sec6():
    hdr("[6] 참신성 십분위별 인용 (cit_z): 시기 비교")
    E = D[D.has_emb & D.cit_z.notna() & D.novelty_prior5.notna() & D.year.between(2000, 2022)].copy()
    E["dec"] = np.ceil(E.groupby("year").novelty_prior5.rank(pct=True) * 10).clip(1, 10).astype(int)
    E["era"] = np.where(E.year <= 2012, "2000-12", "2013-22"); E["hit"] = E.cit_z > 1.28
    t = E.groupby(["era", "dec"]).agg(N=("cit_z", "size"), mean=("cit_z", "mean"), sd=("cit_z", "std"), hit=("hit", "mean")).round(3)
    print(t.unstack(0).to_string())
    for e, g in E.groupby("era"): print(f"{e}: Spearman(novelty, cit_z) {spearmanr(g.novelty_prior5, g.cit_z)[0]:.3f}")
    RES["nov_hit_top_minus_mid"] = E[(E.era == "2013-22") & (E.dec == 10)].hit.mean() - E[(E.era == "2013-22") & E.dec.isin([5, 6])].hit.mean()
    print("※ SPECTER2가 인용으로 학습된 모델이라 참신성과 인용의 관계에는 구조적 상관이 섞입니다. 주 분석용이 아니라 참고용입니다.")
safe("sec6", sec6)

# ============================================================
# [7] 판정표 (임계값은 제 임의 기준)
# ============================================================
hdr("[7] 판정 요약")
g = lambda k: RES.get(k, np.nan)
rows = [
 ["C 구성 견고성", "주제 유효수 변화율(전체)", f"{g('effN_chg_전체'):+.1%}", "고정저널과 부호 동일", np.sign(g('effN_chg_전체')) == np.sign(g('effN_chg_고정저널'))],
 ["C 구성 견고성", "저널-주제 NMI 변화(전체/고정)", f"{g('nmi_chg_전체'):+.3f} / {g('nmi_chg_고정저널'):+.3f}", "부호 동일", np.sign(g('nmi_chg_전체')) == np.sign(g('nmi_chg_고정저널'))],
 ["B 주제 회전", "신생 주제 수", f"{g('n_emerging'):.0f}", ">= 5", g('n_emerging') >= 5],
 ["B 주제 회전", "쇠퇴 주제 수", f"{g('n_declining'):.0f}", ">= 5", g('n_declining') >= 5],
 ["A 갱신 메커니즘", "지속 저자 수", f"{g('n_cont'):,.0f}", ">= 3,000", g('n_cont') >= 3000],
 ["A 갱신 메커니즘", "전환 vs 교체 기여(전환/교체)", f"{g('conv_share'):.0%} / {g('repl_share'):.0%}", "한쪽이 70%+면 서사 선명", max(g('conv_share'), g('repl_share')) >= .7],
 ["네트워크", "거대 성분 비중 변화", f"{g('lcc_first'):.2f} → {g('lcc_last'):.2f}", "변화 >= 0.1", abs(g('lcc_last') - g('lcc_first')) >= .1]]
V = pd.DataFrame(rows, columns=["방향", "지표", "값", "기준", "통과"]); print(V.to_string(index=False)); V.to_csv(P4 / "verdict.csv", index=False, encoding="utf-8-sig")
print("\n완료. 이 셀의 결과로 가설을 세우면 탐색과 확증이 섞입니다. 방향이 정해지면 기간을 나눠 설계와 검정을 분리하세요.")

논문 111,685 | topic150 보유 78.9% | 고정 저널 15개 / 전체 34개

[1] 코퍼스 구성 (기간별)
      papers  venues(>=20편)  임베딩 보유  고정저널 비중  평균 팀규모  단독저자
p                                                         
1990    4318             10   0.593    1.000   2.189 0.307
1995    5564             14   0.656    1.000   2.302 0.279
2000    6298             15   0.729    1.000   2.559 0.197
2005    9103             18   0.789    0.972   2.770 0.152
2010   13601             23   0.828    0.929   3.127 0.100
2015   20163             28   0.867    0.866   3.461 0.066
2020   42831             34   0.852    0.707   4.172 0.034
※ 임베딩 보유율이 기간별로 달라서 이후 주제 분석은 2000년 이후만 씁니다.

[2] 주제 다양성·회전·저널 분화 (전체 / 고정 저널)
 set  period     N  eff_topics   HHI  JS_vs_prev  venue_topic_NMI
  전체    2000  4593      93.704 0.013         NaN            0.132
  전체    2005  7182     104.343 0.011       0.021            0.139
  전체    2010 11266     115.327 0.010       0.024            0.165
  전체    2015 17486     125.062 0.009       0.030        

In [ ]:
# # 결과 해석과 방향 확정
# ## 1. 판정: 방향 A(갱신 메커니즘)를 주축으로 하되, 지금 수치는 그대로 믿으면 안 됩니다
# 판정표는 전부 통과했지만, 임계값이 느슨해서 통과 자체는 의미가 작습니다. 실제로 의미 있는 것은 세 가지입니다.
# | 발견 | 수치 | 읽는 법 |
# |---|---|---|
# | 교체 우세 | 주 분석 전환 17% / 교체 83%, 견고성 18% / 82% | 두 기간 설정에서 거의 같습니다. 이 일관성이 가장 좋은 신호입니다. |
# | 구성 견고성 | 주제 유효수 +32.8%, NMI 증가가 전체와 고정 저널(15개) 모두에서 같은 방향 | 분화가 저널 추가의 산물이라는 의심은 크게 줄었습니다. |
# | 주제 회전 | 신생 41개, 쇠퇴 34개, 연속 기간 순위상관 0.91 → 0.33 | 회전이 가속됩니다. 대표 단어도 해석 가능합니다(COVID, 충전, 마이크로모빌리티, 도로 설계 쇠퇴). |

# ## 2. 그대로 주장하면 안 되는 이유
# **83% 교체는 구조적으로 부풀려질 수 있습니다.** 이 분해에는 큰 문제가 둘 있습니다.
# - **전환 성분의 가중치가 작습니다.** 지속 저자는 P1 논문 질량의 24.5%뿐이라 `conv = w1c × (S1c − S0c)`가 처음부터 작게 나옵니다. 코퍼스가 3배 이상 팽창한 분야라면 "신규가 많다"는 것은 거의 동어반복입니다. 질문은 "교체가 더 큰가"가 아니라 "팽창을 감안해도 교체가 변화를 만드는가"여야 합니다.
# - **지속 저자의 정의가 전환을 과소 측정합니다.** 두 기간 모두 논문이 있는 저자만 지속으로 잡았습니다. 주제를 완전히 바꾼 저자는 같은 이름으로 남지만, 분야를 떠난 저자는 퇴출로 분류됩니다. 또 5년 구간에 논문이 없었던 저자는 "신규"가 됩니다. 신규 75.5% 중 진짜 첫 논문이 2015년 이후인 비율이 73.1%이므로 이 문제는 작지만, 이름 기반 식별 오류는 남습니다.
# 신생 주제 참여율은 오히려 전환 쪽 증거입니다. 지속 저자의 46.8%가 신생 주제에 참여하고(그중 P0에 신생 주제가 없던 저자의 신규 진입이 41.5%), 신규 저자는 53.7%입니다. 이 차이는 7%p뿐입니다. 즉 **기존 저자도 신생 주제로 상당히 이동합니다.** 전체 이동량에서는 교체가 크지만, 신생 주제 채택 행동으로 보면 두 집단이 비슷합니다. 두 지표가 다른 이야기를 하고 있어서, "교체가 지배한다"는 단순 서사는 위험합니다.
# **그 외 점검할 것**
# - 신생 주제 상위가 전기기계(halbach, cogging, pmsm, 주제 88·45)와 텐서·그래프(69)입니다. 이것들이 교통 연구의 변화인지, T-ITS·T-TE 같은 전기·전자 계열 저널의 비중 증가에 따른 코퍼스 구성 효과인지 구분해야 합니다. 고정 저널 15개로 [4]를 다시 돌려야 합니다.
# - 연속 순위상관 0.327은 2015→2020 구간이고, 이 구간에 COVID가 겹칩니다.
# - 참신성-인용 관계는 사실상 없습니다(Spearman −0.02). 이 방향은 접으세요.
# - 네트워크 LCC 0.16 → 0.74는 대부분 팽창과 코퍼스 구성(34개 저널로 점차 확장)의 결과일 가능성이 큽니다. 기술 지표로만 쓰세요.
# - 주제 구조 변화는 `topic150` 해상도와 임베딩 보유율(0.59 → 0.85)에 의존합니다.

# ## 3. 탑저널급으로 가기 위한 설계
# 핵심 질문을 이렇게 바꾸는 것이 좋습니다.
# > **"분야는 사람이 적응해서 변하는가, 사람이 바뀌어서 변하는가?"** (과학의 갱신: Planck 가설의 분야 단위 검정)
# 기존 문헌(과학자의 주제 전환 대 세대 교체)과 직접 연결되고, 일반적인 질문입니다. 다만 선행 연구는 제가 확인하지 못했으므로 별도 검색이 필요합니다. 필요한 보완은 다음과 같습니다.
# 1. **분해를 팽창에 대해 정규화:** 신규 저자 비중이 같은 가상 시나리오(신규가 퇴출과 같은 주제 분포를 가진다는 귀무 모형)와 비교해 "초과 교체"를 측정합니다. 코호트(첫 논문 연도)별로 진입 시점의 주제 분포가 얼마나 다른지, 이후 얼마나 이동하는지를 코호트-연도 패널로 보는 것이 더 깨끗합니다.
# 2. **저자 내 이동의 직접 측정:** 저자별 주제 분포 이동(예: 임베딩 centroid 거리, 이미 있는 `jump`)을 경력 단계별로 보고, 신생 주제 진입 시점의 경력을 봅니다.
# 3. **견고성:** 고정 저널 15개, 두 기간 설정(이미 일치), 동명이인 위험군 제외 기준 변경, 가능하면 기존 `works_by_doi.jsonl` 캐시의 저자 ID(약 5만 편)로 부표본 점검.
# 4. **탐색-확증 분리:** 2015년 이전으로 지표와 가설을 확정하고 이후 기간으로 검정하세요.

# ## 4. 솔직한 평가
# - 이것은 **기술적·구조적 연구**입니다. 인과 식별이 없어서 Science·Nature급보다는 과학학 전문 저널(Research Policy, Scientometrics 상위, JASIST, QSS)이나 분야 저널의 상위가 현실적입니다. 탑 일반 저널을 노린다면 일반성(타 분야 비교)이 필요한데, 외부 수집을 하지 않으면 어렵습니다.
# - 83/17은 헤드라인으로 쓰기엔 아직 이릅니다. 정규화 후에도 교체가 우세하고 코호트가 진입 후 거의 이동하지 않는다는 것이 확인돼야 "교체 중심 갱신"이라고 쓸 수 있습니다. 반대로 정규화 후 차이가 줄면 "기존 저자의 적응이 생각보다 크다"는 반대 서사가 됩니다. 두 결과 모두 쓸 수 있다는 점이 이 설계의 장점입니다.

# ## 5. 다음 셀 제안
# 아래를 한 셀로 만들겠습니다.
# - 코호트(첫 논문 5년 구간)별 진입 시점 주제 분포, 신생 주제 점유율, 진입 후 이동량
# - 교체 귀무 모형 대비 초과 교체 지수(순열: 저자 라벨을 코호트 내에서 섞기)
# - 고정 저널 15개 한정 재실행 및 전기기계 주제(88, 45 등) 제외 재실행
# - 경력 단계별 신생 주제 진입 확률
# 원하시면 이 셀을 만들어 드리겠습니다.

In [7]:
# ============================================================
# 갱신 메커니즘 심화 (단일 셀, 외부 수집 없음)
# [1] 신생 주제 정의(dev: 2005-09 vs 2015-19)
# [2] 전환·교체 초과분 검정 (귀무: 라벨 순열/기간 뒤집기) x 6개 설정
# [3] 코호트별 진입 시점 주제 차이와 진입 후 이동 (매칭 무작위 부표본 대비)
# [4] 경력 단계별 신생 주제 첫 진입 hazard (2020-25, 확증)
# 입력 : analysis_data/{papers_master, paper_ml_features, paper_author_long}.csv
# 출력 : analysis_data/profile5/*.csv
# ============================================================
import warnings, traceback
import numpy as np, pandas as pd, scipy.sparse as sp
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
OUT = Path("./analysis_data"); P5 = OUT / "profile5"; P5.mkdir(exist_ok=True)
rng = np.random.default_rng(0); NPERM = 300
hdr = lambda t: print("\n" + "=" * 95 + f"\n{t}\n" + "=" * 95)
def safe(name, fn):
    try: fn()
    except Exception: print(f"✗ [{name}] 실패"); traceback.print_exc()

# ---------------- 데이터 ----------------
pm = pd.read_csv(OUT / "papers_master.csv", usecols=["pid", "year", "venue_short", "in_sample"])
pm = pm[pm.in_sample.astype(bool) & (pm.year <= 2025)]
ft = pd.read_csv(OUT / "paper_ml_features.csv", usecols=["pid", "topic150"])
pal = pd.read_csv(OUT / "paper_author_long.csv", usecols=["pid", "name_norm", "match_status", "in_sample"], low_memory=False)
pal = pal[pal.in_sample.astype(bool) & pal.name_norm.notna()].drop_duplicates(["pid", "name_norm"])
pal = pal.merge(pm[["pid", "year", "venue_short"]], on="pid").merge(ft, on="pid", how="left")
amb = set(pal.loc[pal.match_status == "ambiguous", "name_norm"])
heavy = set(pal.groupby(["name_norm", "year"]).size().groupby("name_norm").max().loc[lambda s: s >= 12].index)
BASE = pal[~pal.name_norm.isin(amb | heavy)].drop(columns=["match_status", "in_sample"]).copy()
K = int(ft.topic150.max()) + 1
print(f"저자-논문 {len(BASE):,} | 저자 {BASE.name_norm.nunique():,} | 위험군 제외 {len(amb | heavy):,} | K={K}")

# 고정 저널(모든 5년 구간 50편 이상)
pp = pm[pm.year >= 2000].assign(p=lambda d: np.minimum(d.year // 5 * 5, 2020)).groupby(["venue_short", "p"]).size().unstack(fill_value=0)
FIX = set(pp.index[(pp[[2000, 2005, 2010, 2015, 2020]] >= 50).all(axis=1)]); print(f"고정 저널 {len(FIX)}개")

def js(p, q):
    p, q = p / p.sum(), q / q.sum(); m = .5 * (p + q)
    k = lambda a, b: np.sum(a[a > 0] * np.log(a[a > 0] / b[a > 0]))
    return .5 * k(p, m) + .5 * k(q, m)

def prep(venues=None, excl=()):
    d = BASE if venues is None else BASE[BASE.venue_short.isin(venues)]
    first = d.groupby("name_norm").year.min()                       # 경력 시작은 제외 전 전체 기록 기준
    t = d.dropna(subset=["topic150"]).copy(); t = t[~t.topic150.isin(excl)]
    t["topic150"] = t.topic150.astype(int)
    t["w"] = 1.0 / t.groupby("pid").name_norm.transform("size")
    t["a"], names = pd.factorize(t.name_norm)
    return t, first.reindex(names).values.astype(int)

def tsh(t, y0, y1):
    s = t[t.year.between(y0, y1)].drop_duplicates("pid")
    c = np.bincount(s.topic150, minlength=K).astype(float); return c / c.sum()

def sparse_mat(t, y0, y1, nA):
    s = t[t.year.between(y0, y1)]
    return sp.csr_matrix((s.w.values, (s.a.values, s.topic150.values)), shape=(nA, K))

# ============================================================
# [1] 신생 주제 (dev 정의: 2005-09 vs 2015-19 → 2020-25에서 검정)
# ============================================================
T0, F0 = prep()
def emerg(t, a, b, ratio=3, minsh=.005):
    s0, s1 = tsh(t, *a), tsh(t, *b); r = (s1 + 1e-4) / (s0 + 1e-4)
    return set(np.where((r >= ratio) & (s1 >= minsh))[0])
EM_DEV, EM_FULL = emerg(T0, (2005, 2009), (2015, 2019)), emerg(T0, (2005, 2009), (2020, 2025))
hdr("[1] 신생 주제 정의")
print(f"EM_dev(2005-09→2015-19) {len(EM_DEV)}개 | EM_full(2005-09→2020-25) {len(EM_FULL)}개 | 교집합 {len(EM_DEV & EM_FULL)}개")
print(f"EM_dev 주제의 2020-25 논문 비중: {tsh(T0, 2020, 2025)[list(EM_DEV)].sum():.1%} (2005-09: {tsh(T0, 2005, 2009)[list(EM_DEV)].sum():.1%})")

# ============================================================
# [2] 전환·교체 초과분 (귀무 모형)
# ============================================================
def core(t, p0, p1):
    nA = t.a.max() + 1
    X0s, X1s = sparse_mat(t, *p0, nA), sparse_mat(t, *p1, nA)
    m0, m1 = X0s.sum(1).A1, X1s.sum(1).A1; act = (m0 > 0) | (m1 > 0)
    X0, X1 = X0s[act].toarray(), X1s[act].toarray(); a0, a1 = m0[act] > 0, m1[act] > 0
    cont, ent, ext = a0 & a1, a1 & ~a0, a0 & ~a1
    S = lambda X, m: X[m].sum(0)
    r = dict(n_cont=int(cont.sum()), n_ent=int(ent.sum()), n_ext=int(ext.sum()), w1c=S(X1, cont).sum() / X1.sum())
    # (a) 전환: 지속 저자의 P0→P1 이동, 귀무=저자별 기간 라벨 무작위 뒤집기
    Xc0, Xc1 = X0[cont], X1[cont]; Dm = Xc1 - Xc0; b0, b1 = Xc0.sum(0), Xc1.sum(0)
    r["conv"] = js(b0, b1); nl = []
    for _ in range(NPERM):
        fd = (rng.random(len(Dm)) < .5).astype(float) @ Dm; nl.append(js(b0 + fd, b1 - fd))
    r["conv_null"], r["conv_sd"] = np.mean(nl), np.std(nl)
    # (b) gap: 같은 P1 기간에서 신규 vs 지속 (시대 효과 제거), 귀무=라벨 순열
    Xg = np.vstack([X1[ent], X1[cont]]); lab = np.r_[np.ones(ent.sum()), np.zeros(cont.sum())]; tot = Xg.sum(0)
    r["gap"] = js(S(X1, ent), S(X1, cont)); nl = []
    for _ in range(NPERM):
        v = rng.permutation(lab) @ Xg; nl.append(js(v, tot - v))
    r["gap_null"], r["gap_sd"] = np.mean(nl), np.std(nl)
    # (c) 교체: 신규(P1) vs 퇴출(P0)
    Xu = np.vstack([X1[ent], X0[ext]]); lab = np.r_[np.ones(ent.sum()), np.zeros(ext.sum())]; tot = Xu.sum(0)
    r["repl"] = js(S(X1, ent), S(X0, ext)); nl = []
    for _ in range(NPERM):
        v = rng.permutation(lab) @ Xu; nl.append(js(v, tot - v))
    r["repl_null"] = np.mean(nl)
    r["S0"], r["S1"] = X0.sum(0), X1.sum(0)
    return r

def sec2():
    hdr("[2] 전환·교체 초과분 (excess = 관측 JS − 귀무 평균 JS, z = excess / 귀무 sd)")
    P_MAIN, P_ALT = ((2010, 2014), (2020, 2024)), ((2005, 2009), (2015, 2019))
    cfgs = [("1 주 분석", None, (), P_MAIN), ("2 대체 기간", None, (), P_ALT), ("3 고정저널", FIX, (), P_MAIN),
            ("4 전기기계(88,45) 제외", None, (88, 45), P_MAIN), ("5 고정저널+제외", FIX, (88, 45), P_MAIN),
            ("6 고정저널+대체기간", FIX, (), P_ALT)]
    rows = []
    for name, ven, ex, (p0, p1) in cfgs:
        t, _ = prep(ven, ex); r = core(t, p0, p1)
        rows.append(dict(설정=name, 지속=r["n_cont"], 신규=r["n_ent"], 퇴출=r["n_ext"], 지속저자질량=r["w1c"],
                         전환_JS=r["conv"], 전환_excess=r["conv"] - r["conv_null"], 전환_z=(r["conv"] - r["conv_null"]) / r["conv_sd"],
                         gap_JS=r["gap"], gap_excess=r["gap"] - r["gap_null"], gap_z=(r["gap"] - r["gap_null"]) / r["gap_sd"],
                         교체_excess=r["repl"] - r["repl_null"],
                         gap대전환=(r["gap"] - r["gap_null"]) / max(r["conv"] - r["conv_null"], 1e-9)))
    V = pd.DataFrame(rows); print(V.to_string(index=False)); V.to_csv(P5 / "excess_replacement.csv", index=False)
    print("\n해석: gap대전환 = (같은 시점 신규-지속 차이의 초과분) / (지속 저자 자신의 이동 초과분).")
    print("  1보다 크게 안정적이면 '코호트 차이가 개인 이동보다 큼'(교체 중심), 1 안팎이면 비슷, 1보다 작으면 적응 중심.")
    print("  z가 2~3 이하인 지표는 표본 잡음과 구분되지 않습니다.")
safe("sec2", sec2)

# ============================================================
# [3] 코호트: 진입 시점 주제 차이와 진입 후 이동
# ============================================================
def sec3():
    hdr("[3] 코호트 분석 (주 설정, 위험군 제외, 이름 기반)")
    t, first = T0, F0; nA = t.a.max() + 1
    coh = np.where(first < 2000, 1995, np.minimum(first // 5 * 5, 2020))                  # 1995 = '2000 이전 진입'
    EMl = np.array(sorted(EM_DEV))
    print("--- (a) 기간별·코호트별: 질량 비중, EM_dev 주제 비중, 나머지 코호트 대비 JS(초과분=무작위 동규모 부표본 대비) ---")
    rows = []
    for p0, p1 in [(2010, 2014), (2015, 2019), (2020, 2025)]:
        X = sparse_mat(t, p0, p1, nA); m = X.sum(1).A1; act = np.where(m > 0)[0]; Xd = X[act].toarray(); cc = coh[act]; tot = Xd.sum(0)
        for c in sorted(set(cc)):
            sel = cc == c; n = int(sel.sum())
            if n < 200 or c > p0: continue
            v = Xd[sel].sum(0); obs = js(v, tot - v)
            nl = []
            for _ in range(20):
                idx = rng.choice(len(act), n, replace=False); u = Xd[idx].sum(0); nl.append(js(u, tot - u))
            rows.append(dict(period=f"{p0}-{p1}", cohort=("<2000" if c == 1995 else c), authors=n, mass_share=v.sum() / tot.sum(),
                             em_share=v[EMl].sum() / v.sum(), em_share_period=tot[EMl].sum() / tot.sum(),
                             JS_vs_rest=obs, JS_excess=obs - np.mean(nl)))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(P5 / "cohort_by_period.csv", index=False)
    print("\n--- (b) 진입 코호트의 진입 후 이동: 진입기 → 2020-25 (두 기간 모두 활동한 저자만) ---")
    XL = sparse_mat(t, 2020, 2025, nA); rows = []
    for c in (2000, 2005, 2010):
        XE = sparse_mat(t, c, c + 4, nA); me, ml = XE.sum(1).A1, XL.sum(1).A1
        sel = np.where((coh == c) & (me > 0) & (ml > 0))[0]
        if len(sel) < 100: continue
        E, L = XE[sel].toarray(), XL[sel].toarray(); b0, b1 = E.sum(0), L.sum(0); Dm = L - E; obs = js(b0, b1); nl = []
        for _ in range(NPERM):
            fd = (rng.random(len(sel)) < .5).astype(float) @ Dm; nl.append(js(b0 + fd, b1 - fd))
        fa = js(sparse_mat(t, c, c + 4, nA).sum(0).A1, XL.sum(0).A1)
        rows.append(dict(cohort=c, 지속=len(sel), 코호트_이동JS=obs, 귀무=np.mean(nl), excess=obs - np.mean(nl),
                         z=(obs - np.mean(nl)) / np.std(nl), 분야전체_JS=fa, em_진입기=E[:, EMl].sum() / E.sum(), em_2020_25=L[:, EMl].sum() / L.sum()))
    B = pd.DataFrame(rows); print(B.to_string(index=False)); B.to_csv(P5 / "cohort_movement.csv", index=False)
    print("해석: 코호트_이동JS가 분야전체_JS보다 훨씬 작으면 '기존 저자는 덜 움직이고 분야는 새 코호트로 이동'을 뒷받침합니다.")
    print("※ 지속 저자는 오래 활동한 선택된 표본이고, 분야전체_JS는 모든 저자의 구성 변화를 포함합니다.")
safe("sec3", sec3)

# ============================================================
# [4] 경력 단계별 신생 주제 첫 진입 (EM_dev 정의 → 2020-25 검정)
# ============================================================
def sec4():
    hdr("[4] 신생 주제 첫 진입 hazard (2020-25, 정의는 2015-19까지의 자료로 고정)")
    t, first = T0.copy(), F0
    t["em"] = t.topic150.isin(EM_DEV).astype(int)
    g = t.groupby(["a", "year"]).agg(em=("em", "max"), n=("pid", "size")).reset_index().sort_values(["a", "year"])
    g["prior"] = g.groupby("a").em.cumsum() - g.em
    g["career"] = g.year - first[g.a.values]
    R = g[(g.prior == 0) & g.year.between(2020, 2025)].copy()
    R["cg"] = pd.cut(R.career, [-1, 0, 5, 15, 100], labels=["0(첫해)", "1-5", "6-15", ">15"])
    print(f"위험집합 {len(R):,} 저자-연도 | 이벤트 {int(R.em.sum()):,}\n")
    print(R.groupby("cg").em.agg(위험="size", 이벤트="sum", 비율="mean").to_string())
    print("\n연도별 비율:\n", R.pivot_table(index="cg", columns="year", values="em", aggfunc="mean").round(3).to_string())
    try:
        import statsmodels.formula.api as smf
        R["lny"] = np.log(R.n); R["cgs"] = R.cg.astype(str)
        m = smf.logit("em ~ C(cgs, Treatment('1-5')) + lny + C(year)", data=R).fit(disp=0, cov_type="cluster", cov_kwds={"groups": R.a.values})
        k = [i for i in m.params.index if "cgs" in i or i == "lny"]
        print("\n로짓 (기준 경력 1-5년, 연도 FE, 저자 클러스터):")
        print(pd.DataFrame({"OR": np.exp(m.params[k]), "p": m.pvalues[k]}).round(3).to_string())
    except Exception as e: print("로짓 생략:", repr(e))
    print("※ 경력 0(첫해)은 신규 진입자입니다. 1년 이상 경력군이 '전환', 0이 '교체'에 해당합니다. 단일 논문 해에는 노출량 통제(lny)가 중요합니다.")
safe("sec4", sec4)
print("\n완료. 이 셀의 결과는 탐색 단계의 확증 검정입니다(정의는 2015-19까지, 평가는 2020-25).")

저자-논문 335,988 | 저자 140,553 | 위험군 제외 1,431 | K=150
고정 저널 15개

[1] 신생 주제 정의
EM_dev(2005-09→2015-19) 14개 | EM_full(2005-09→2020-25) 42개 | 교집합 13개
EM_dev 주제의 2020-25 논문 비중: 16.6% (2005-09: 2.2%)

[2] 전환·교체 초과분 (excess = 관측 JS − 귀무 평균 JS, z = excess / 귀무 sd)
              설정   지속    신규    퇴출  지속저자질량  전환_JS  전환_excess     전환_z  gap_JS  gap_excess    gap_z  교체_excess  gap대전환
          1 주 분석 5065 45458 12183  0.2386 0.0710     0.0676 169.2926  0.0577      0.0537 111.2263     0.1585  0.7953
         2 대체 기간 3142 24360  7228  0.2653 0.0430     0.0374  58.7116  0.0271      0.0209  27.1499     0.1095  0.5591
          3 고정저널 4502 32873 11874  0.2677 0.0682     0.0645 152.9901  0.0309      0.0266  49.2506     0.1238  0.4117
4 전기기계(88,45) 제외 5058 43977 12188  0.2454 0.0709     0.0674 173.1165  0.0476      0.0438  85.9843     0.1479  0.6494
       5 고정저널+제외 4502 32858 11874  0.2677 0.0682     0.0645 140.5175  0.0308      0.0265  51.2290     0.1237  0.4110
     6 고정저널+대체기간 2977 21208  7206  0.2732 0.

In [8]:
#

In [9]:
# ============================================================
# 동명이인 견고성 + 저널 내부 분해 (단일 셀, 외부 수집 없음)
# [1] 이름 위험군 변형 7종 x 2기간  [2] 이름 vs OpenAlex 저자ID(캐시) 직접 비교
# [3] 저널 구성 효과 분해: gap / 전환의 저널 내부 성분  [4] 판정
# 입력 : analysis_data/{papers_master, paper_ml_features, paper_author_long}.csv, analysis_data/openalex/works_by_doi.jsonl
# 출력 : analysis_data/profile6/*.csv
# ============================================================
import json, warnings, traceback
import numpy as np, pandas as pd, scipy.sparse as sp
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
OUT = Path("./analysis_data"); P6 = OUT / "profile6"; P6.mkdir(exist_ok=True)
rng = np.random.default_rng(0); NPERM = 100
hdr = lambda t: print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)
def safe(n, f):
    try: f()
    except Exception: print(f"✗ [{n}] 실패"); traceback.print_exc()
RES = {}

# ---------------- 데이터 ----------------
pm = pd.read_csv(OUT / "papers_master.csv", usecols=["pid", "year", "venue_short", "in_sample", "doi_norm"])
pm = pm[pm.in_sample.astype(bool) & (pm.year <= 2025)]
ft = pd.read_csv(OUT / "paper_ml_features.csv", usecols=["pid", "topic150"])
raw = pd.read_csv(OUT / "paper_author_long.csv", usecols=["pid", "author_pos", "name_norm", "match_status", "in_sample"], low_memory=False)
raw = raw[raw.in_sample.astype(bool)].drop(columns="in_sample")
raw = raw.merge(pm[["pid", "year", "venue_short"]], on="pid").merge(ft, on="pid", how="left")
pal = raw[raw.name_norm.notna()].drop_duplicates(["pid", "name_norm"]).copy()
K = int(ft.topic150.max()) + 1
P_MAIN, P_ALT = ((2010, 2014), (2020, 2024)), ((2005, 2009), (2015, 2019))

# ---------------- 공통 함수 ----------------
def js(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    if p.sum() <= 0 or q.sum() <= 0: return np.nan
    p, q = p / p.sum(), q / q.sum(); m = .5 * (p + q)
    k = lambda a, b: np.sum(a[a > 0] * np.log(a[a > 0] / b[a > 0]))
    return .5 * k(p, m) + .5 * k(q, m)

def prep(d, key):
    t = d.dropna(subset=["topic150"]).copy(); t["topic150"] = t.topic150.astype(int)
    t["w"] = 1.0 / t.groupby("pid")[key].transform("size")
    t["a"] = pd.factorize(t[key])[0]; t["j"] = pd.factorize(t.venue_short)[0]
    return t

def core(t, p0, p1, nperm=NPERM):
    nA = t.a.max() + 1
    def mat(y0, y1):
        s = t[t.year.between(y0, y1)]
        return sp.csr_matrix((s.w.values, (s.a.values, s.topic150.values)), shape=(nA, K))
    X0s, X1s = mat(*p0), mat(*p1); m0, m1 = X0s.sum(1).A1, X1s.sum(1).A1; act = (m0 > 0) | (m1 > 0)
    X0, X1 = X0s[act].toarray(), X1s[act].toarray(); a0, a1 = m0[act] > 0, m1[act] > 0
    cont, ent = a0 & a1, a1 & ~a0
    if cont.sum() < 30 or ent.sum() < 30: return None
    r = dict(n_cont=int(cont.sum()), n_ent=int(ent.sum()))
    Dm = X1[cont] - X0[cont]; b0, b1 = X0[cont].sum(0), X1[cont].sum(0); r["conv"] = js(b0, b1); nl = []
    for _ in range(nperm):
        fd = (rng.random(len(Dm)) < .5).astype(float) @ Dm; nl.append(js(b0 + fd, b1 - fd))
    r["conv_x"] = r["conv"] - np.mean(nl)
    Xg = np.vstack([X1[ent], X1[cont]]); lab = np.r_[np.ones(ent.sum()), np.zeros(cont.sum())]; tot = Xg.sum(0)
    r["gap"] = js(X1[ent].sum(0), X1[cont].sum(0)); nl = []
    for _ in range(nperm):
        v = rng.permutation(lab) @ Xg; nl.append(js(v, tot - v))
    r["gap_x"] = r["gap"] - np.mean(nl); r["ratio"] = r["gap_x"] / max(r["conv_x"], 1e-9)
    return r

# ============================================================
# [1] 이름 위험군 변형
# ============================================================
def sec1():
    hdr("[1] 동명이인 위험군 기준 변형 (excess = 관측 JS - 귀무 평균 JS)")
    amb = set(pal.loc[pal.match_status == "ambiguous", "name_norm"])
    mx = pal.groupby(["name_norm", "year"]).size().groupby("name_norm").max()
    g = pal.groupby("name_norm").year; span = g.max() - g.min()
    nm = pd.Series(pal.name_norm.unique()); ft_len = nm.str.split().str[0].str.len(); ntok = nm.str.split().str.len()
    init, short2 = set(nm[ft_len == 1]), set(nm[(ntok == 2) & (nm.str.len() <= 10)])
    h = lambda k: set(mx[mx >= k].index); wide = set(span[span > 35].index)
    V = {"0 기준(연12+, 모호)": amb | h(12), "1 연8+": amb | h(8), "2 연5+": amb | h(5),
         "3 기준+이니셜 이름": amb | h(12) | init, "4 기준+경력폭>35년": amb | h(12) | wide,
         "5 기준+짧은 2토큰 이름": amb | h(12) | short2, "6 전부 결합": amb | h(5) | init | wide | short2}
    rows = []
    for nme, S in V.items():
        d = pal[~pal.name_norm.isin(S)]; t = prep(d, "name_norm")
        for lab, (p0, p1) in [("주", P_MAIN), ("대체", P_ALT)]:
            r = core(t, p0, p1)
            if r: rows.append(dict(변형=nme, 기간=lab, 제외이름=len(S), **r))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(P6 / "name_variants.csv", index=False)
    for lab in ["주", "대체"]:
        a = T[T.기간 == lab].set_index("변형")
        RES[f"conv_keep_{lab}"] = a.conv_x.iloc[-1] / a.conv_x.iloc[0]; RES[f"gap_keep_{lab}"] = a.gap_x.iloc[-1] / a.gap_x.iloc[0]
    print("\n※ '짧은 2토큰'은 동명이인 가능성이 높은 이름을 거칠게 잡는 대리 기준이라 실제 인물도 제거합니다. 방향 점검용입니다.")
safe("sec1", sec1)

# ============================================================
# [2] 이름 vs 저자 ID (캐시, 2020-2025 동일 논문·동일 기간)
# ============================================================
def sec2():
    hdr("[2] 이름 기반 vs OpenAlex 저자 ID (같은 논문, 2020-22 → 2023-25)")
    fa = OUT / "openalex" / "works_by_doi.jsonl"
    O = pd.DataFrame([{"doi_norm": (r.get("doi_norm") or "").lower(), "ids": r.get("authors") or [], "n_oa": r.get("n_oa_auth")}
                      for r in map(json.loads, fa.open(encoding="utf-8"))]).drop_duplicates("doi_norm")
    pmd = pm.assign(doi_norm=pm.doi_norm.str.lower()).dropna(subset=["doi_norm"]).drop_duplicates("doi_norm")[["pid", "doi_norm"]]
    O = O.merge(pmd, on="doi_norm"); nrows = raw.groupby("pid").size()
    O["n_rows"] = O.pid.map(nrows); ok = (O.ids.map(len) == O.n_oa) & (O.n_oa == O.n_rows)
    Ok = O[ok]; print(f"캐시 매칭 논문 {len(O):,} | 저자 수 정합(위치 정렬 가능) {len(Ok):,} | 2020-25 in_sample 대비 커버리지 {len(Ok)/pm[pm.year >= 2020].shape[0]:.1%}")
    L = pd.DataFrame({"pid": np.repeat(Ok.pid.values, Ok.ids.map(len).values), "oa_id": [x for l in Ok.ids for x in l],
                      "rank": np.concatenate([np.arange(len(l)) for l in Ok.ids])})
    R = raw.sort_values(["pid", "author_pos"]).copy(); R["rank"] = R.groupby("pid").cumcount()
    M = R.merge(L, on=["pid", "rank"]); M = M[M.year.between(2020, 2025)]
    amb = set(pal.loc[pal.match_status == "ambiguous", "name_norm"])
    heavy = set(pal.groupby(["name_norm", "year"]).size().groupby("name_norm").max().loc[lambda s: s >= 12].index)
    sets = {"이름(위험군 제외)": M[M.name_norm.notna() & ~M.name_norm.isin(amb | heavy)].drop_duplicates(["pid", "name_norm"]).assign(key=lambda d: d.name_norm),
            "이름(제외 없음)": M[M.name_norm.notna()].drop_duplicates(["pid", "name_norm"]).assign(key=lambda d: d.name_norm),
            "OpenAlex 저자 ID": M.drop_duplicates(["pid", "oa_id"]).assign(key=lambda d: d.oa_id)}
    rows = []
    for nme, d in sets.items():
        t = prep(d, "key"); r = core(t, (2020, 2022), (2023, 2025))
        if r: rows.append(dict(식별=nme, 저자수=t.a.nunique(), **r))
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(P6 / "name_vs_id.csv", index=False)
    if len(T) == 3:
        RES["id_over_name"] = T.conv_x.iloc[2] / T.conv_x.iloc[0]; RES["id_gap_over_name"] = T.gap_x.iloc[2] / T.gap_x.iloc[0]
    print("\n※ 기간이 짧아(3년 대 3년) 절대값은 앞 결과와 다릅니다. 이름과 ID의 상대 비교만 보세요. ID도 병합·분리 오류가 있어 정답이 아니라 독립된 두 번째 측정입니다.")
safe("sec2", sec2)

# ============================================================
# [3] 저널 구성 효과 분해
# ============================================================
def pair_js(vA, vB, J, minmass=30):
    A, B = vA.reshape(J, K), vB.reshape(J, K); ok = (A.sum(1) >= minmass) & (B.sum(1) >= minmass)
    if ok.sum() < 2: return np.nan, np.nan
    A, B = A[ok], B[ok]; wB = B.sum(1) / B.sum()
    A_adj = ((A / A.sum(1, keepdims=True)) * wB[:, None]).sum(0)       # A를 B의 저널 구성으로 재가중
    return js(A.sum(0), B.sum(0)), js(A_adj, B.sum(0))

def within(t, p0, p1):
    nA, J = t.a.max() + 1, t.j.max() + 1
    def mat(y0, y1):
        s = t[t.year.between(y0, y1)]
        return sp.csr_matrix((s.w.values, (s.a.values, s.j.values * K + s.topic150.values)), shape=(nA, J * K))
    X0, X1 = mat(*p0), mat(*p1); m0, m1 = X0.sum(1).A1, X1.sum(1).A1
    cont, ent = np.where((m0 > 0) & (m1 > 0))[0], np.where((m1 > 0) & (m0 == 0))[0]
    out = {}
    Xg = sp.vstack([X1[ent], X1[cont]]).tocsr(); lab = np.r_[np.ones(len(ent)), np.zeros(len(cont))]
    a, b = Xg.T @ lab, Xg.T @ (1 - lab); tot, adj = pair_js(a, b, J); nt, na = [], []
    for _ in range(NPERM):
        l = rng.permutation(lab); x, y = pair_js(Xg.T @ l, Xg.T @ (1 - l), J); nt.append(x); na.append(y)
    out["gap"] = (tot - np.nanmean(nt), adj - np.nanmean(na))
    Dm = (X1[cont] - X0[cont]).tocsr(); v0, v1 = X0[cont].sum(0).A1, X1[cont].sum(0).A1; tot, adj = pair_js(v0, v1, J); nt, na = [], []
    for _ in range(NPERM):
        fd = Dm.T @ (rng.random(len(cont)) < .5).astype(float); x, y = pair_js(v0 + fd, v1 - fd, J); nt.append(x); na.append(y)
    out["conv"] = (tot - np.nanmean(nt), adj - np.nanmean(na))
    return out, len(cont), len(ent)

def sec3():
    hdr("[3] 저널 구성 분해: 전체 excess vs 저널 구성을 맞춘 뒤(=저널 내부) excess")
    amb = set(pal.loc[pal.match_status == "ambiguous", "name_norm"])
    heavy = set(pal.groupby(["name_norm", "year"]).size().groupby("name_norm").max().loc[lambda s: s >= 12].index)
    t = prep(pal[~pal.name_norm.isin(amb | heavy)], "name_norm"); rows = []
    for lab, (p0, p1) in [("주", P_MAIN), ("대체", P_ALT)]:
        o, nc, ne = within(t, p0, p1)
        for k, nm in [("gap", "gap(신규 vs 지속)"), ("conv", "전환(지속 저자 이동)")]:
            tot, adj = o[k]; rows.append(dict(기간=lab, 지표=nm, 지속=nc, 신규=ne, 전체_excess=tot, 저널내부_excess=adj,
                                              저널내부_비중=adj / tot, 저널구성_기여=1 - adj / tot))
            RES[f"{k}_within_{lab}"] = adj / tot
    T = pd.DataFrame(rows); print(T.to_string(index=False)); T.to_csv(P6 / "journal_decomp.csv", index=False)
    print("\n해석: 저널내부_비중이 높으면 신호가 저널 안의 하위 주제 차이/이동입니다. 낮으면 저널 간 이동·구성 효과입니다.")
    print("※ 양쪽 모두 질량 30 이상인 저널만 비교하고, 저널 안의 하위 분야 이질성은 남습니다.")
safe("sec3", sec3)

# ============================================================
# [4] 판정 (임계값은 제 임의 기준)
# ============================================================
hdr("[4] 판정 요약")
g = lambda k: RES.get(k, np.nan)
V = pd.DataFrame([
 ["동명이인", "전환 excess 유지율(가장 엄격/기준, 주·대체)", f"{g('conv_keep_주'):.0%} / {g('conv_keep_대체'):.0%}", ">= 70%", min(g('conv_keep_주'), g('conv_keep_대체')) >= .7],
 ["동명이인", "ID 기반 전환 excess / 이름 기반", f"{g('id_over_name'):.0%}", ">= 60%", g('id_over_name') >= .6],
 ["저널 내부", "전환의 저널내부 비중(주/대체)", f"{g('conv_within_주'):.0%} / {g('conv_within_대체'):.0%}", ">= 70%", min(g('conv_within_주'), g('conv_within_대체')) >= .7],
 ["저널 내부", "gap의 저널내부 비중(주/대체)", f"{g('gap_within_주'):.0%} / {g('gap_within_대체'):.0%}", "참고(낮으면 교체=구성효과)", np.nan]],
 columns=["영역", "지표", "값", "기준", "통과"])
print(V.to_string(index=False)); V.to_csv(P6 / "verdict.csv", index=False, encoding="utf-8-sig")
print("\n세 통과 지표가 모두 True면 '저널 내부의 개인 적응'을 주결과로 계속 진행, 하나라도 False면 해당 약점부터 해결하거나 데이터 자원 논문 경로로 전환하세요.")


[1] 동명이인 위험군 기준 변형 (excess = 관측 JS - 귀무 평균 JS)
            변형 기간  제외이름  n_cont  n_ent   conv  conv_x    gap  gap_x  ratio
0 기준(연12+, 모호)  주  1431    5065  45458 0.0710  0.0675 0.0577 0.0537 0.7949
0 기준(연12+, 모호) 대체  1431    3142  24360 0.0430  0.0374 0.0271 0.0207 0.5531
         1 연8+  주  1755    4859  45349 0.0674  0.0635 0.0562 0.0520 0.8195
         1 연8+ 대체  1755    3006  24220 0.0412  0.0348 0.0282 0.0215 0.6169
         2 연5+  주  3010    4294  44795 0.0635  0.0584 0.0503 0.0453 0.7745
         2 연5+ 대체  3010    2619  23694 0.0393  0.0307 0.0310 0.0230 0.7480
   3 기준+이니셜 이름  주  8700    4990  44812 0.0707  0.0673 0.0579 0.0540 0.8026
   3 기준+이니셜 이름 대체  8700    3064  23836 0.0432  0.0375 0.0271 0.0207 0.5527
  4 기준+경력폭>35년  주  1815    4911  45395 0.0721  0.0684 0.0559 0.0517 0.7556
  4 기준+경력폭>35년 대체  1815    2966  24300 0.0433  0.0374 0.0255 0.0188 0.5032
5 기준+짧은 2토큰 이름  주 25892    4385  34168 0.0720  0.0677 0.0597 0.0549 0.8117
5 기준+짧은 2토큰 이름 대체 25892    2859  20375 0.0448  0.038

In [10]:
#

In [11]:
# ============================================================
# 확증 분할 + 메커니즘 분해 (단일 셀, 외부 수집 없음)
# [1] 새 기간 쌍 + 단일 저널 한정 재현   [2] 전환의 주체·방향 분해   [3] 판정
# 입력 : analysis_data/{papers_master, paper_ml_features, paper_author_long}.csv
# 출력 : analysis_data/profile7/*.csv
# ============================================================
import warnings, traceback
import numpy as np, pandas as pd, scipy.sparse as sp
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
OUT = Path("./analysis_data"); P7 = OUT / "profile7"; P7.mkdir(exist_ok=True)
rng = np.random.default_rng(0); NPERM = 100
hdr = lambda t: print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)
def safe(n, f):
    try: f()
    except Exception: print(f"✗ [{n}] 실패"); traceback.print_exc()
RES = {}

# ---------------- 데이터 ----------------
pm = pd.read_csv(OUT / "papers_master.csv", usecols=["pid", "year", "venue_short", "in_sample", "n_authors"])
pm = pm[pm.in_sample.astype(bool) & (pm.year <= 2025)]
ft = pd.read_csv(OUT / "paper_ml_features.csv", usecols=["pid", "topic150"])
pal = pd.read_csv(OUT / "paper_author_long.csv", usecols=["pid", "name_norm", "match_status", "in_sample"], low_memory=False)
pal = pal[pal.in_sample.astype(bool) & pal.name_norm.notna()].drop_duplicates(["pid", "name_norm"])
pal = pal.merge(pm[["pid", "year", "venue_short", "n_authors"]], on="pid").merge(ft, on="pid", how="left")
amb = set(pal.loc[pal.match_status == "ambiguous", "name_norm"])
heavy = set(pal.groupby(["name_norm", "year"]).size().groupby("name_norm").max().loc[lambda s: s >= 12].index)
BASE = pal[~pal.name_norm.isin(amb | heavy)].copy()
FIRST = BASE.groupby("name_norm").year.min()                    # 경력 시작(주제 보유 여부와 무관)
K = int(ft.topic150.max()) + 1
P_MAIN = ((2010, 2014), (2020, 2024))
print(f"저자-논문 {len(BASE):,} | 저자 {BASE.name_norm.nunique():,} | K={K}")

# ---------------- 공통 함수 ----------------
def js(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    if p.sum() <= 0 or q.sum() <= 0: return np.nan
    p, q = p / p.sum(), q / q.sum(); m = .5 * (p + q)
    k = lambda a, b: np.sum(a[a > 0] * np.log(a[a > 0] / b[a > 0]))
    return .5 * k(p, m) + .5 * k(q, m)

def prep(d):
    t = d.dropna(subset=["topic150"]).copy(); t["topic150"] = t.topic150.astype(int)
    t["w"] = 1.0 / t.groupby("pid").name_norm.transform("size")
    t["a"], names = pd.factorize(t.name_norm); t["j"] = pd.factorize(t.venue_short)[0]
    return t, names

def conv_excess(X0, X1, nperm=NPERM):                           # 지속 저자 집합의 전환 excess
    Dm = X1 - X0; b0, b1 = X0.sum(0), X1.sum(0); obs = js(b0, b1); nl = []
    for _ in range(nperm):
        fd = (rng.random(len(Dm)) < .5).astype(float) @ Dm; nl.append(js(b0 + fd, b1 - fd))
    return obs - np.nanmean(nl)

def core(t, p0, p1):
    nA = t.a.max() + 1
    def mat(y0, y1):
        s = t[t.year.between(y0, y1)]
        return sp.csr_matrix((s.w.values, (s.a.values, s.topic150.values)), shape=(nA, K))
    X0s, X1s = mat(*p0), mat(*p1); m0, m1 = X0s.sum(1).A1, X1s.sum(1).A1; act = (m0 > 0) | (m1 > 0)
    X0, X1 = X0s[act].toarray(), X1s[act].toarray(); a0, a1 = m0[act] > 0, m1[act] > 0
    cont, ent = a0 & a1, a1 & ~a0
    if cont.sum() < 30 or ent.sum() < 30: return None
    r = dict(n_cont=int(cont.sum()), n_ent=int(ent.sum())); r["conv_x"] = conv_excess(X0[cont], X1[cont])
    Xg = np.vstack([X1[ent], X1[cont]]); lab = np.r_[np.ones(ent.sum()), np.zeros(cont.sum())]; tot = Xg.sum(0)
    obs = js(X1[ent].sum(0), X1[cont].sum(0)); nl = []
    for _ in range(NPERM):
        v = rng.permutation(lab) @ Xg; nl.append(js(v, tot - v))
    r["gap_x"] = obs - np.mean(nl); r["ratio"] = r["gap_x"] / max(r["conv_x"], 1e-9)
    return r

def pair_js(vA, vB, J, minmass=30):                             # 저널 구성을 B에 맞춘 JS
    A, B = vA.reshape(J, K), vB.reshape(J, K); ok = (A.sum(1) >= minmass) & (B.sum(1) >= minmass)
    if ok.sum() < 2: return np.nan, np.nan
    A, B = A[ok], B[ok]; wB = B.sum(1) / B.sum()
    return js(A.sum(0), B.sum(0)), js(((A / A.sum(1, keepdims=True)) * wB[:, None]).sum(0), B.sum(0))

def within(t, p0, p1):
    nA, J = t.a.max() + 1, t.j.max() + 1
    def mat(y0, y1):
        s = t[t.year.between(y0, y1)]
        return sp.csr_matrix((s.w.values, (s.a.values, s.j.values * K + s.topic150.values)), shape=(nA, J * K))
    X0, X1 = mat(*p0), mat(*p1); m0, m1 = X0.sum(1).A1, X1.sum(1).A1
    cont, ent = np.where((m0 > 0) & (m1 > 0))[0], np.where((m1 > 0) & (m0 == 0))[0]
    if len(cont) < 30 or len(ent) < 30: return None
    out = {}
    Xg = sp.vstack([X1[ent], X1[cont]]).tocsr(); lab = np.r_[np.ones(len(ent)), np.zeros(len(cont))]
    tot, adj = pair_js(Xg.T @ lab, Xg.T @ (1 - lab), J); nt, na = [], []
    for _ in range(NPERM):
        l = rng.permutation(lab); x, y = pair_js(Xg.T @ l, Xg.T @ (1 - l), J); nt.append(x); na.append(y)
    out["gap"] = (tot - np.nanmean(nt), adj - np.nanmean(na))
    Dm = (X1[cont] - X0[cont]).tocsr(); v0, v1 = X0[cont].sum(0).A1, X1[cont].sum(0).A1
    tot, adj = pair_js(v0, v1, J); nt, na = [], []
    for _ in range(NPERM):
        fd = Dm.T @ (rng.random(len(cont)) < .5).astype(float); x, y = pair_js(v0 + fd, v1 - fd, J); nt.append(x); na.append(y)
    out["conv"] = (tot - np.nanmean(nt), adj - np.nanmean(na))
    return out

T_ALL, NAMES = prep(BASE)

# ============================================================
# [1] 확증 분할
# ============================================================
def sec1():
    hdr("[1a] 새 기간 쌍 (이전 단계에서 쓰지 않은 설정) — 예측: conv_x>0, 전환 저널내부>=70%, gap 저널내부<=25%")
    pairs = {"A 2000-04→2010-14": ((2000, 2004), (2010, 2014)), "B 1995-99→2005-09": ((1995, 1999), (2005, 2009)),
             "C 3년창 2012-14→2022-24": ((2012, 2014), (2022, 2024)), "D 2000-04→2020-24(20년)": ((2000, 2004), (2020, 2024)),
             "E 참고: 2010-14→2020-24(기존 주분석)": P_MAIN}
    rows = []
    for nm, (p0, p1) in pairs.items():
        r, w = core(T_ALL, p0, p1), within(T_ALL, p0, p1)
        if r is None or w is None: continue
        rows.append(dict(설정=nm, 지속=r["n_cont"], 신규=r["n_ent"], conv_x=r["conv_x"], gap_x=r["gap_x"], ratio=r["ratio"],
                         전환_저널내부=w["conv"][1] / w["conv"][0], gap_저널내부=w["gap"][1] / w["gap"][0]))
    V = pd.DataFrame(rows); print(V.to_string(index=False)); V.to_csv(P7 / "confirm_periods.csv", index=False)
    new = V[~V.설정.str.startswith("E")]
    RES["p_conv"] = (new.conv_x > 0).all(); RES["p_cw"] = (new.전환_저널내부 >= .7).mean(); RES["p_gw"] = (new.gap_저널내부 <= .25).mean()

    hdr("[1b] 단일 저널 한정 (저널 구성 효과가 구조적으로 없음) — 주 기간 2010-14→2020-24")
    top = BASE[BASE.year.between(2010, 2024)].venue_short.value_counts().head(8).index; rows = []
    for v in top:
        t, _ = prep(BASE[BASE.venue_short == v]); r = core(t, *P_MAIN)
        if r: rows.append(dict(저널=v, **r))
    S = pd.DataFrame(rows); print(S.to_string(index=False)); S.to_csv(P7 / "confirm_single_venue.csv", index=False)
    RES["sv_pos"] = (S.conv_x > 0).mean() if len(S) else np.nan; RES["sv_n"] = len(S)
    print("해석: 단일 저널 안에서도 conv_x > 0이고 gap_x가 conv_x보다 작으면 '저널 내부 적응 > 저널 내부 신규-지속 차이'가 재현된 것입니다.")
safe("sec1", sec1)

# ============================================================
# [2] 메커니즘: 누가, 어디로 이동하는가 (2010-14 → 2020-24, 지속 저자)
# ============================================================
def emerg(t, a, b, lo, hi):
    f = lambda y: (lambda s: np.bincount(s.topic150, minlength=K) / len(s))(t[t.year.between(*y)].drop_duplicates("pid"))
    s0, s1 = f(a), f(b); r = (s1 + 1e-4) / (s0 + 1e-4)
    return np.where((r >= 3) & (s1 >= .005))[0], np.where((r <= 1 / 3) & (s0 >= .005))[0]
EM, DEC = emerg(T_ALL, (2005, 2009), (2015, 2019), 3, 1 / 3)     # 2020 이후 자료를 보지 않고 정의

def sec2():
    hdr(f"[2] 지속 저자의 이동 분해 | 신생 주제 {len(EM)}개, 쇠퇴 주제 {len(DEC)}개 (2005-09 vs 2015-19로 정의)")
    t, names = T_ALL, NAMES; nA = t.a.max() + 1; (p0, p1) = P_MAIN
    def mat(y0, y1):
        s = t[t.year.between(y0, y1)]
        return sp.csr_matrix((s.w.values, (s.a.values, s.topic150.values)), shape=(nA, K))
    X0s, X1s = mat(*p0), mat(*p1); m0, m1 = X0s.sum(1).A1, X1s.sum(1).A1
    ids = np.where((m0 > 0) & (m1 > 0))[0]; X0, X1 = X0s[ids].toarray(), X1s[ids].toarray()
    D = pd.DataFrame({"a": ids, "name": names[ids], "m0": X0.sum(1), "m1": X1.sum(1)})
    D["em0"], D["em1"] = X0[:, EM].sum(1) / D.m0, X1[:, EM].sum(1) / D.m1
    D["dc0"], D["dc1"] = X0[:, DEC].sum(1) / D.m0, X1[:, DEC].sum(1) / D.m1
    D["dem"], D["ddec"] = D.em1 - D.em0, D.dc1 - D.dc0
    D["tv"] = .5 * np.abs(X1 / D.m1.values[:, None] - X0 / D.m0.values[:, None]).sum(1)
    f = FIRST.reindex(D.name).values; D["cohort"] = np.select([f < 2000, f < 2005, f < 2010], ["<2000", "2000-04", "2005-09"], "2010-14")
    ps = t[t.year.between(*p0)].drop_duplicates("pid").set_index("pid")
    tm = t[t.year.between(*p0)].groupby("name_norm").n_authors.mean(); D["team0"] = tm.reindex(D.name).values
    mv = t[t.year.between(*p0)].groupby(["name_norm", "venue_short"]).w.sum().reset_index().sort_values("w", ascending=False).drop_duplicates("name_norm")
    D["mainv"] = mv.set_index("name_norm").venue_short.reindex(D.name).values
    # 공저자 노출: P0 공저자의 (ego 공저 논문을 제외한) P0 신생 주제 비중
    b = BASE[BASE.year.between(*p0)]; ego = b[b.name_norm.isin(D.name)][["pid", "name_norm"]].rename(columns={"name_norm": "ego"})
    pr = ego.merge(b[["pid", "name_norm"]].rename(columns={"name_norm": "alter"}), on="pid"); pr = pr[pr.ego != pr.alter]
    ap = b.dropna(subset=["topic150"])[["name_norm", "pid", "topic150"]].rename(columns={"name_norm": "alter", "pid": "pid2"})
    ap["em"] = ap.topic150.astype(int).isin(EM).astype(int)
    x = pr[["ego", "alter"]].drop_duplicates().merge(ap, on="alter")
    x = x.merge(ego.rename(columns={"pid": "pid2"}).assign(own=1), on=["ego", "pid2"], how="left"); x = x[x.own.isna()]
    ce = x.groupby("ego").em.mean(); D["coem"] = D.name.map(ce); D["coem_miss"] = D.coem.isna().astype(int); D["coem"] = D.coem.fillna(0)
    D["lnco"] = np.log1p(D.name.map(pr.groupby("ego").alter.nunique()).fillna(0))
    D["lm0"], D["lm1"] = np.log(D.m0), np.log(D.m1)
    D.to_csv(P7 / "continuing_authors.csv", index=False)

    print(f"지속 저자 {len(D):,} | 평균 신생 비중 P0 {D.em0.mean():.3f} → P1 {D.em1.mean():.3f} | 쇠퇴 비중 {D.dc0.mean():.3f} → {D.dc1.mean():.3f}")
    print("\n--- (a) 방향성: 지속 저자 합산 분포 이동 중 신생/쇠퇴 주제의 몫 vs 분야 전체 ---")
    b0, b1 = X0.sum(0), X1.sum(0); d = b1 / b1.sum() - b0 / b0.sum()
    allp0, allp1 = X0s.sum(0).A1, X1s.sum(0).A1; da = allp1 / allp1.sum() - allp0 / allp0.sum()
    print(f"지속 저자 Δ(신생) {d[EM].sum():+.3f}, Δ(쇠퇴) {d[DEC].sum():+.3f} | 분야 전체 Δ(신생) {da[EM].sum():+.3f}, Δ(쇠퇴) {da[DEC].sum():+.3f}")
    RES["cont_em_ratio"] = d[EM].sum() / da[EM].sum()
    print(f"→ 지속 저자는 분야 전체 신생 주제 증가의 {RES['cont_em_ratio']:.0%} 수준을 따라갑니다(질량 {D.m1.sum() / allp1.sum():.0%}만 차지하는데도).")

    print("\n--- (b) 부분집단별 전환 excess (귀무 대비), 평균 신생비중 변화 ---")
    rows = []
    def grp(label, key, series):
        for g, idx in D.groupby(series).groups.items():
            ii = D.index.get_indexer(idx)
            if len(ii) < 100: continue
            rows.append(dict(기준=label, 집단=g, 저자수=len(ii), 평균_P0질량=D.m0.values[ii].mean(), conv_excess=conv_excess(X0[ii], X1[ii]),
                             평균_Δ신생=D.dem.values[ii].mean(), 평균_Δ쇠퇴=D.ddec.values[ii].mean(), 평균_TV=D.tv.values[ii].mean()))
    grp("코호트", "cohort", D.cohort)
    grp("P0 팀규모", "team0", pd.qcut(D.team0, 3, labels=["소", "중", "대"], duplicates="drop"))
    grp("생산성(P0+P1)", "m", pd.qcut(D.m0 + D.m1, 3, labels=["하", "중", "상"], duplicates="drop"))
    grp("공저자 신생노출", "coem", pd.qcut(D.coem.rank(method="first"), 3, labels=["하", "중", "상"]))
    G = pd.DataFrame(rows); print(G.to_string(index=False)); G.to_csv(P7 / "conv_by_group.csv", index=False)
    print("※ excess는 표본 크기 편향을 귀무로 보정하지만, 집단 간 P0·P1 질량 차이가 크면 비교에 주의하세요.")

    print("\n--- (c) 개인 단위 회귀: 종속 Δ신생비중 / TV (저널 클러스터) ---")
    try:
        import statsmodels.formula.api as smf
        Z = D.dropna(subset=["team0", "mainv"]).copy(); Z["team0"] = np.log(Z.team0.clip(lower=1))
        for dv in ["dem", "tv"]:
            m = smf.ols(f"{dv} ~ C(cohort, Treatment('2010-14')) + lm0 + lm1 + team0 + em0 + coem + coem_miss + lnco + C(mainv)", data=Z) \
                   .fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(Z.mainv)[0]})
            k = [i for i in m.params.index if "cohort" in i or i in ("lm0", "lm1", "team0", "em0", "coem", "lnco")]
            print(f"\n[{dv}] N={int(m.nobs):,}, R2={m.rsquared:.3f}"); print(pd.DataFrame({"coef": m.params[k], "se": m.bse[k], "p": m.pvalues[k]}).round(4).to_string())
            if dv == "dem": RES["coem_coef"], RES["coem_p"] = m.params["coem"], m.pvalues["coem"]
    except Exception as e: print("회귀 생략:", repr(e))
    print("\n※ 공저자 노출은 상관이지 영향의 증거가 아닙니다. 같은 하위 분야의 저자가 함께 이동(선택/동질성)하는 효과와 구분되지 않습니다.")
    print("※ em0(출발 비중)은 평균회귀 통제용이고, 신생 주제는 정의상 P0에서 낮아 Δ신생이 양으로 치우칩니다.")
safe("sec2", sec2)

# ============================================================
# [3] 판정 (임계값은 제 임의 기준)
# ============================================================
hdr("[3] 판정 요약")
g = lambda k: RES.get(k, np.nan)
V = pd.DataFrame([
 ["확증(새 기간 쌍)", "전환 excess > 0 (모든 새 설정)", str(g("p_conv")), "True", bool(g("p_conv"))],
 ["확증(새 기간 쌍)", "전환 저널내부 >= 70% 비율", f"{g('p_cw'):.0%}", "100%", g("p_cw") == 1],
 ["확증(새 기간 쌍)", "gap 저널내부 <= 25% 비율", f"{g('p_gw'):.0%}", "100%", g("p_gw") == 1],
 ["확증(단일 저널)", "conv_x > 0 인 저널 비율", f"{g('sv_pos'):.0%} (n={g('sv_n'):.0f})", ">= 80%", g("sv_pos") >= .8],
 ["메커니즘", "지속 저자의 신생 주제 증가 / 분야 전체", f"{g('cont_em_ratio'):.0%}", "참고", np.nan],
 ["메커니즘", "공저자 신생노출 계수(p)", f"{g('coem_coef'):.3f} ({g('coem_p'):.3f})", "참고", np.nan]],
 columns=["영역", "지표", "값", "기준", "통과"])
print(V.to_string(index=False)); V.to_csv(P7 / "verdict.csv", index=False, encoding="utf-8-sig")
print("\n완료. 확증 4개가 모두 True면 '저널 내부 개인 적응' 서사를 유지하고, 하나라도 False면 서사를 그 설정에 한정해 쓰세요.")

저자-논문 335,988 | 저자 140,553 | K=150

[1a] 새 기간 쌍 (이전 단계에서 쓰지 않은 설정) — 예측: conv_x>0, 전환 저널내부>=70%, gap 저널내부<=25%
                           설정   지속    신규  conv_x  gap_x  ratio  전환_저널내부  gap_저널내부
            A 2000-04→2010-14 1984 15264  0.0244 0.0134 0.5501   0.8139   -0.1188
            B 1995-99→2005-09 1357  9013  0.0222 0.0107 0.4826   1.0873   15.0040
        C 3년창 2012-14→2022-24 3435 35302  0.0697 0.0614 0.8808   0.9224    0.1432
       D 2000-04→2020-24(20년) 1317 49206  0.1131 0.0675 0.5968   0.8638   -0.0193
E 참고: 2010-14→2020-24(기존 주분석) 5065 45458  0.0675 0.0537 0.7959   0.8754    0.0989

[1b] 단일 저널 한정 (저널 구성 효과가 구조적으로 없음) — 주 기간 2010-14→2020-24
   저널  n_cont  n_ent  conv_x   gap_x   ratio
  TRR    1327   6987  0.0533  0.0154  0.2882
T-ITS     454  11912  0.1304  0.0090  0.0693
  AAP     524   3493  0.0358  0.0208  0.5820
 TR-D     164   4036  0.0590  0.0135  0.2293
 TR-C     253   3198  0.0674 -0.0128 -0.1896
 TR-F     234   2761  0.0231  0.0006  0.0239
 TR-A     235   2672  0

In [12]:
#